In [6]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from collections import Counter
import time

qc_base = QuantumCircuit.from_qasm_file("P6_low_hill.qasm")
print(f"Qubits: {qc_base.num_qubits}, Depth: {qc_base.depth()}")

# approximate transpilation at 0.99
print("\nTranspiling at 0.99...")
qc_tr = transpile(
    qc_base,
    basis_gates=list(qc_base.count_ops().keys()),
    approximation_degree=0.99,
    optimization_level=3,
)

before = sum(qc_base.count_ops().values())
after = sum(qc_tr.count_ops().values())
print(f"Gates removed: {before - after} ({(before-after)/before*100:.1f}% reduction)")

qc_tr.measure_all()

simulator = AerSimulator(
    method="matrix_product_state",
    matrix_product_state_max_bond_dimension=256,
)

all_counts = Counter()
start = time.time()

for i in range(3):
    print(f"\nRun {i+1}/3...")
    job = simulator.run(qc_tr, shots=1024)
    counts = job.result().get_counts()
    all_counts.update(counts)
    peak = max(all_counts, key=all_counts.get)
    total = sum(all_counts.values())
    print(f"confidence: {all_counts[peak]/total:.4%} | Time: {time.time()-start:.0f}s")
    print("Top 3:")
    for bits, count in sorted(all_counts.items(), key=lambda x: -x[1])[:3]:
        print(f"  {bits}: {count/total:.4%}")

total = sum(all_counts.values())
peak = max(all_counts, key=all_counts.get)
bit_counts = [Counter() for _ in range(len(peak))]
for bitstring, count in all_counts.items():
    for j, bit in enumerate(bitstring):
        bit_counts[j][bit] += count
majority_peak = ''.join(c.most_common(1)[0][0] for c in bit_counts)

print(f"\n{'='*60}")
print(f"Direct peak:\n{peak}")
print(f"Confidence: {all_counts[peak]/total:.4%}")
print(f"\nMajority vote peak:\n{majority_peak}")
print(f"{'='*60}")

print("\nTop 10:")
for bits, count in sorted(all_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {bits}: {count/total:.4%}")

Qubits: 60, Depth: 139

Transpiling at 0.99...
Gates removed: 2055 (95.7% reduction)

Run 1/3...
confidence: 99.4141% | Time: 0s
Top 3:
  111011000000000001110101110001101001111011100010100101001101 111011000000000001110101110001101001111011100010100101001101: 99.4141%
  101011000000000001110101110001101001111011100010100101001101 101011000000000001110101110001101001111011100010100101001101: 0.3906%
  111011000000000001110101110001101001111011100000100101001101 111011000000000001110101110001101001111011100000100101001101: 0.1953%

Run 2/3...
confidence: 99.4141% | Time: 0s
Top 3:
  111011000000000001110101110001101001111011100010100101001101 111011000000000001110101110001101001111011100010100101001101: 99.4141%
  101011000000000001110101110001101001111011100010100101001101 101011000000000001110101110001101001111011100010100101001101: 0.3906%
  111011000000000001110101110001101001111011100000100101001101 111011000000000001110101110001101001111011100000100101001101: 0.1465%

Run 3/3...
c